# 02 Layout Aware Document Parsing

**Project:** Enterprise Multi-Document RAG Assistant

**Notebook:** `02-layout-aware-document-parsing.ipynb`

In [ ]:
# Start coding here

In [36]:
# !pip install pandas
# !pip install pdfplumber
# !pip install unstructured

In [37]:
# =====================================================
# Notebook 02
# Layout Aware Document Parsing
# =====================================================

import json
import os

import pandas as pd

In [38]:
with open("artifacts/document_repository.json", "r", encoding="utf-8") as file:

    repository = json.load(file)

print(f"Loaded {len(repository)} documents")

Loaded 3 documents


In [39]:
sample_document = """

# Annual Financial Report 2026

## Executive Summary

Revenue reached $5.2 billion.

Operating margin improved to 18%.

## Regional Revenue Table

Region | Revenue
US | 2.1B
Europe | 1.7B
Asia | 1.4B

## Cloud Division

Cloud business grew 35%.

AI products generated record profits.

"""

In [40]:
print(sample_document)



# Annual Financial Report 2026

## Executive Summary

Revenue reached $5.2 billion.

Operating margin improved to 18%.

## Regional Revenue Table

Region | Revenue
US | 2.1B
Europe | 1.7B
Asia | 1.4B

## Cloud Division

Cloud business grew 35%.

AI products generated record profits.




In [41]:
element_schema = {
    "element_id": "",
    "document_id": "",
    "element_type": "",
    "content": "",
    "page": 1,
    "section": "",
}

element_schema

{'element_id': '',
 'document_id': '',
 'element_type': '',
 'content': '',
 'page': 1,
 'section': ''}

In [42]:
def extract_titles(text):

    titles = []

    lines = text.split("\n")

    for line in lines:

        if line.startswith("# "):

            titles.append(line.replace("# ", "").strip())

    return titles

In [43]:
extract_titles(sample_document)

['Annual Financial Report 2026']

In [44]:
def extract_headings(text):

    headings = []

    lines = text.split("\n")

    for line in lines:

        if line.startswith("## "):

            headings.append(line.replace("## ", "").strip())

    return headings

In [45]:
extract_headings(sample_document)

['Executive Summary', 'Regional Revenue Table', 'Cloud Division']

In [46]:
def extract_tables(text):

    tables = []

    lines = text.split("\n")

    for line in lines:

        if "|" in line:

            tables.append(line.strip())

    return tables

In [47]:
extract_tables(sample_document)

['Region | Revenue', 'US | 2.1B', 'Europe | 1.7B', 'Asia | 1.4B']

In [48]:
def extract_paragraphs(text):

    paragraphs = []

    lines = text.split("\n")

    for line in lines:

        line = line.strip()

        if len(line) < 10:

            continue

        if line.startswith("#"):

            continue

        if "|" in line:

            continue

        paragraphs.append(line)

    return paragraphs

In [49]:
extract_paragraphs(sample_document)

['Revenue reached $5.2 billion.',
 'Operating margin improved to 18%.',
 'Cloud business grew 35%.',
 'AI products generated record profits.']

In [50]:
import uuid


def parse_document(text, document_id):

    elements = []

    titles = extract_titles(text)

    headings = extract_headings(text)

    paragraphs = extract_paragraphs(text)

    tables = extract_tables(text)

    for title in titles:

        elements.append(
            {
                "element_id": str(uuid.uuid4()),
                "document_id": document_id,
                "element_type": "title",
                "content": title,
            }
        )

    for heading in headings:

        elements.append(
            {
                "element_id": str(uuid.uuid4()),
                "document_id": document_id,
                "element_type": "heading",
                "content": heading,
            }
        )

    for paragraph in paragraphs:

        elements.append(
            {
                "element_id": str(uuid.uuid4()),
                "document_id": document_id,
                "element_type": "paragraph",
                "content": paragraph,
            }
        )

    for table in tables:

        elements.append(
            {
                "element_id": str(uuid.uuid4()),
                "document_id": document_id,
                "element_type": "table",
                "content": table,
            }
        )

    return elements

In [51]:
parsed_elements = parse_document(sample_document, "finance_doc")

In [69]:
elements_df = pd.DataFrame(parsed_elements)

elements_df.head(100)

,element_id,document_id,element_type,content
0,b9f65a29-d348-492b-8e95-b429c2dd7241,finance_doc,title,Annual Financial Report 2026
1,d150c61d-5a49-43d0-ac4d-e4dd4543cfc5,finance_doc,heading,Executive Summary
2,2419d783-0424-4753-8c23-61615a9d3be0,finance_doc,heading,Regional Revenue Table
3,7ec26f07-f891-4969-9586-7154d4b26073,finance_doc,heading,Cloud Division
4,58b319f4-790d-4de3-9e82-75fac00621da,finance_doc,paragraph,Revenue reached $5.2 billion.
5,234b8659-9f97-42b3-85e2-e1a30c700807,finance_doc,paragraph,Operating margin improved to 18%.
6,0cee15c7-9b76-4835-9203-f6d1845db7b9,finance_doc,paragraph,Cloud business grew 35%.
7,a84bfce7-bf59-45b8-8c17-da1416061df3,finance_doc,paragraph,AI products generated record profits.
8,d6897837-d9e0-4c25-9dea-c18e338a2276,finance_doc,table,Region | Revenue
9,676bff4d-7a6c-4067-9389-52ea16a9ff51,finance_doc,table,US | 2.1B


In [53]:
all_elements = []

for doc_id, doc in repository.items():

    parsed = parse_document(doc["content"], doc_id)

    all_elements.extend(parsed)

In [54]:
parsed_df = pd.DataFrame(all_elements)

parsed_df.head(100)

,element_id,document_id,element_type,content
0,ca5c2139-5dfc-4f72-8c9e-bfc08a019543,1fe99400-06e1-4964-956b-3861687e7da4,title,AI Platform Roadmap
1,1bed6371-5f93-4441-b9b5-35fbadab95e6,1fe99400-06e1-4964-956b-3861687e7da4,heading,Search Infrastructure
2,a0edaf4d-ddb1-4bc3-82f9-4d12c633e138,1fe99400-06e1-4964-956b-3861687e7da4,heading,Retrieval Systems
3,d6e86085-e962-4a2a-ba3b-fd335d4b6318,1fe99400-06e1-4964-956b-3861687e7da4,heading,Deployment Roadmap
4,06534449-c490-49cd-8d33-0f908a2214ef,1fe99400-06e1-4964-956b-3861687e7da4,paragraph,Vector search infrastructure rollout.
5,cc020ebf-ff98-4e3d-b652-faf11603c6e8,1fe99400-06e1-4964-956b-3861687e7da4,paragraph,Search latency improved significantly.
6,7a35aafc-7d35-445a-b97e-cab9eab135d3,1fe99400-06e1-4964-956b-3861687e7da4,paragraph,New indexing pipeline deployed.
7,5e3d6d0c-800e-4a42-8e7b-e095deb48dce,1fe99400-06e1-4964-956b-3861687e7da4,paragraph,Hybrid retrieval implementation.
8,13338c72-05ef-48bf-9dca-709a76091108,1fe99400-06e1-4964-956b-3861687e7da4,paragraph,Semantic search quality improved.
9,06af26e4-bb81-4512-bfb9-1687a32f4630,1fe99400-06e1-4964-956b-3861687e7da4,paragraph,Knowledge base coverage expanded.


In [55]:
parsed_df = pd.DataFrame(all_elements)

parsed_df.head()

,element_id,document_id,element_type,content
0,ca5c2139-5dfc-4f72-8c9e-bfc08a019543,1fe99400-06e1-4964-956b-3861687e7da4,title,AI Platform Roadmap
1,1bed6371-5f93-4441-b9b5-35fbadab95e6,1fe99400-06e1-4964-956b-3861687e7da4,heading,Search Infrastructure
2,a0edaf4d-ddb1-4bc3-82f9-4d12c633e138,1fe99400-06e1-4964-956b-3861687e7da4,heading,Retrieval Systems
3,d6e86085-e962-4a2a-ba3b-fd335d4b6318,1fe99400-06e1-4964-956b-3861687e7da4,heading,Deployment Roadmap
4,06534449-c490-49cd-8d33-0f908a2214ef,1fe99400-06e1-4964-956b-3861687e7da4,paragraph,Vector search infrastructure rollout.


In [56]:
parsed_df["element_type"].value_counts()

element_type
paragraph    27
table        12
heading       9
title         3
Name: count, dtype: int64

In [57]:
document_tree = {}

for _, row in parsed_df.iterrows():

    doc_id = row["document_id"]

    if doc_id not in document_tree:

        document_tree[doc_id] = []

    document_tree[doc_id].append(
        {"type": row["element_type"], "content": row["content"]}
    )

In [58]:
first_doc = list(document_tree.keys())[0]

document_tree[first_doc][:5]

[{'type': 'title', 'content': 'AI Platform Roadmap'},
 {'type': 'heading', 'content': 'Search Infrastructure'},
 {'type': 'heading', 'content': 'Retrieval Systems'},
 {'type': 'heading', 'content': 'Deployment Roadmap'},
 {'type': 'paragraph', 'content': 'Vector search infrastructure rollout.'}]

In [59]:
parsed_df.to_csv("artifacts/parsed_elements.csv", index=False)

In [60]:
with open("artifacts/document_tree.json", "w", encoding="utf-8") as file:

    json.dump(document_tree, file, indent=4)

In [61]:
# !pip install pi-heif

In [62]:
# !pip install "unstructured[pdf]"

In [63]:
# !pip install unstructured-inference

In [64]:
# from unstructured.partition.pdf import partition_pdf

# elements = partition_pdf("report.pdf")

In [65]:
# !pip install llama-index llama-parse

In [66]:
# parser = LlamaParse()

# documents = parser.load_data("report.pdf")

In [67]:
for doc_id, doc in repository.items():

    print("=" * 80)
    print(doc_id)
    print("=" * 80)

    print(doc["content"][:1000])

    break

1fe99400-06e1-4964-956b-3861687e7da4

# AI Platform Roadmap

## Search Infrastructure

Vector search infrastructure rollout.

Search latency improved significantly.

New indexing pipeline deployed.

## Retrieval Systems

Hybrid retrieval implementation.

Semantic search quality improved.

Knowledge base coverage expanded.

## Deployment Roadmap

Cross encoder deployment planned.

Enterprise RAG platform launch Q3.

Agentic workflows under evaluation.

Component | Status
Vector Search | Production
Hybrid Retrieval | Testing
Cross Encoder | Planned



In [68]:
for doc_id, doc in repository.items():

    parsed = parse_document(doc["content"], doc_id)

    df = pd.DataFrame(parsed)

    print("\n", doc_id)

    print(df["element_type"].value_counts())


 1fe99400-06e1-4964-956b-3861687e7da4
element_type
paragraph    9
table        4
heading      3
title        1
Name: count, dtype: int64

 b638264b-b788-442e-a09d-19176466fba8
element_type
paragraph    9
table        4
heading      3
title        1
Name: count, dtype: int64

 31516a79-2cf5-4080-ac66-66b338d6ebff
element_type
paragraph    9
table        4
heading      3
title        1
Name: count, dtype: int64
